In [2]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
import time
import matplotlib.pyplot as plt
import json

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

pd.set_option('display.max_columns', 500)

In [3]:
version = '20260711'
download_base = Path('../../data/allen-brain-cell-atlas-staging')
abc_cache = AbcProjectCache.from_cache_dir(
    download_base,
    s3_bucket='allen-brain-cell-atlas-staging',
    auth_required=True,
)

abc_cache.load_manifest(f'releases/{version}/manifest.json')

/Users/chris.morrison/src/abc_atlas_access/src/abc_atlas_access/abc_atlas_cache/cloud_cache.py:679: OutdatedManifestWarning: You are loading releases/20260131/manifest.json. A more up to date version of the dataset -- releases/20260711/manifest.json -- exists online. To see the changes between the two versions of the dataset, run
type.compare_manifests('releases/20260131/manifest.json', 'releases/20260711/manifest.json')
To load another version of the dataset, run
type.load_manifest('releases/20260711/manifest.json')
  warnings.warn(msg, OutdatedManifestWarning)


Read in the two DataFrames from the aging dataset we'll need to create an equivalent cluster annotation terms and term set like the WMB and WHB taxonomies.

In [4]:
taxonomy_dir = 'SEA-AD-Multiregion-taxonomy'

In [5]:
abc_cache.list_metadata_files(taxonomy_dir)

['cell_2d_embedding_coordinates',
 'cell_to_cluster_membership',
 'cluster',
 'cluster_annotation_term',
 'cluster_annotation_term_set',
 'cluster_to_cluster_annotation_membership']

In [6]:
term = abc_cache.get_metadata_dataframe(
    taxonomy_dir,
    'cluster_annotation_term'
)
term

cluster_annotation_term.csv: 100%|██████████| 32.3k/32.3k [00:00<00:00, 248kMB/s]


,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,CCN20230508_label
0,CS20260630_CLAS_001,Neuronal: GABAergic,CCN20260630_LEVEL_0,Class,#F05A28,1,0,NaN,NaN,NaN,NaN
1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,Class,#00ADF8,2,0,NaN,NaN,NaN,NaN
2,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,Class,#808080,3,0,NaN,NaN,NaN,NaN
3,CS20260630_SCLA_023,Astrocyte,CCN20260630_LEVEL_1,Subclass,#665C47,23,1,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,NaN
4,CS20260630_SCLA_021,CA2-4,CCN20260630_LEVEL_1,Subclass,#D5BF41,21,1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
234,CS20260630_SUPR_038,Vip_25-SEAAD,CCN20260630_LEVEL_2,Supertype,#492C56,38,2,CS20260630_SCLA_005,Vip,CCN20260630_LEVEL_1,NaN
235,CS20260630_SUPR_039,Vip_4,CCN20260630_LEVEL_2,Supertype,#D0AEDC,39,2,CS20260630_SCLA_005,Vip,CCN20260630_LEVEL_1,CS20230508_SUPT_0022
236,CS20260630_SUPR_040,Vip_5,CCN20260630_LEVEL_2,Supertype,#C7A5D3,40,2,CS20260630_SCLA_005,Vip,CCN20260630_LEVEL_1,CS20230508_SUPT_0023
237,CS20260630_SUPR_041,Vip_6,CCN20260630_LEVEL_2,Supertype,#BE9DCA,41,2,CS20260630_SCLA_005,Vip,CCN20260630_LEVEL_1,CS20230508_SUPT_0024


In [7]:
term_sets = abc_cache.get_metadata_dataframe(
    directory=taxonomy_dir,
    file_name='cluster_annotation_term_set'
).set_index('label')
term_sets

cluster_annotation_term_set.csv:   0%|          | 0.00/208 [00:00<?, ?MB/s]

cluster_annotation_term_set.csv: 100%|██████████| 208/208 [00:00<00:00, 992MB/s]  


,name,description,order,parent_term_set_label
label,,,,
CCN20260630_LEVEL_0,Class,Class,0,NaN
CCN20260630_LEVEL_1,Subclass,Subclass,1,CCN20260630_LEVEL_0
CCN20260630_LEVEL_2,Supertype,Supertype,2,CCN20260630_LEVEL_1


In [8]:
filtered = term[pd.notna(term['parent_term_label'])]
first_child = filtered.groupby('parent_term_label')[['label','name','term_order','cluster_annotation_term_set_name']].first()
first_child

,label,name,term_order,cluster_annotation_term_set_name
parent_term_label,,,,
CS20260630_CLAS_001,CS20260630_SCLA_009,Chandelier,9,Subclass
CS20260630_CLAS_002,CS20260630_SCLA_021,CA2-4,21,Subclass
CS20260630_CLAS_003,CS20260630_SCLA_023,Astrocyte,23,Subclass
CS20260630_SCLA_001,CS20260630_SUPR_001,Lamp5_Lhx6_1,1,Supertype
CS20260630_SCLA_002,CS20260630_SUPR_005,Lamp5_1,5,Supertype
CS20260630_SCLA_003,CS20260630_SUPR_011,Pax6_1,11,Supertype
CS20260630_SCLA_004,CS20260630_SUPR_018,Sncg_1,18,Supertype
CS20260630_SCLA_005,CS20260630_SUPR_025,Vip_1,25,Supertype
CS20260630_SCLA_006,CS20260630_SUPR_043,Sst Chodl_1,43,Supertype


In [9]:
term.set_index('label',inplace=True)
term.loc[first_child.index,'first_child_label'] = first_child['label']
term.loc[first_child.index,'first_child_term_set_name'] = first_child['cluster_annotation_term_set_name']
term.reset_index(inplace=True)

In [10]:
term[pd.notna(term['first_child_label'])]

,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,CCN20230508_label,first_child_label,first_child_term_set_name
0,CS20260630_CLAS_001,Neuronal: GABAergic,CCN20260630_LEVEL_0,Class,#F05A28,1,0,NaN,NaN,NaN,NaN,CS20260630_SCLA_009,Subclass
1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,Class,#00ADF8,2,0,NaN,NaN,NaN,NaN,CS20260630_SCLA_021,Subclass
2,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,Class,#808080,3,0,NaN,NaN,NaN,NaN,CS20260630_SCLA_023,Subclass
3,CS20260630_SCLA_023,Astrocyte,CCN20260630_LEVEL_1,Subclass,#665C47,23,1,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_175,Supertype
4,CS20260630_SCLA_021,CA2-4,CCN20260630_LEVEL_1,Subclass,#D5BF41,21,1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_169,Supertype
5,CS20260630_SCLA_009,Chandelier,CCN20260630_LEVEL_1,Subclass,#F641A8,9,1,CS20260630_CLAS_001,Neuronal: GABAergic,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_082,Supertype
6,CS20260630_SCLA_022,DG,CCN20260630_LEVEL_1,Subclass,#D8442B,22,1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_173,Supertype
7,CS20260630_SCLA_019,EC IT,CCN20260630_LEVEL_1,Subclass,#73BC48,19,1,CS20260630_CLAS_002,Neuronal: Glutamatergic,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_158,Supertype
8,CS20260630_SCLA_027,Endothelial,CCN20260630_LEVEL_1,Subclass,#8D6C62,27,1,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_192,Supertype
9,CS20260630_SCLA_024,Ependymal,CCN20260630_LEVEL_1,Subclass,#7A5151,24,1,CS20260630_CLAS_003,Non-neuronal and Non-neural,CCN20260630_LEVEL_0,NaN,CS20260630_SUPR_181,Supertype


In [13]:
membership = abc_cache.get_metadata_dataframe(directory=taxonomy_dir, file_name='cluster_to_cluster_annotation_membership')
pivot = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])['cluster_annotation_term_name'].first().unstack()
pivot = pivot[term_sets['name']] # order columns
pivot.fillna('Other', inplace=True)
pivot.sort_values(['Class', 'Subclass', 'Supertype'], inplace=True)
cols = pivot.columns.to_list()
pivot.columns = cols
pivot

,Class,Subclass,Supertype
cluster_alias,,,
82,Neuronal: GABAergic,Chandelier,Chandelier_1
83,Neuronal: GABAergic,Chandelier,Chandelier_2
5,Neuronal: GABAergic,Lamp5,Lamp5_1
6,Neuronal: GABAergic,Lamp5,Lamp5_2
7,Neuronal: GABAergic,Lamp5,Lamp5_3
...,...,...,...
195,Non-neuronal and Non-neural,VLMC & Perivascular,Pericyte_1
196,Non-neuronal and Non-neural,VLMC & Perivascular,Pericyte_2-SEAAD
197,Non-neuronal and Non-neural,VLMC & Perivascular,SMC-SEAAD


In [14]:
lookup = {}
for tag in term_sets['name']:
    #print(tag)
    pred = (term['cluster_annotation_term_set_name'] == tag)
    filtered = term[pred].copy()
    filtered.set_index('name', inplace=True)
    lookup[tag] = filtered

Helper functions to lookup an term attribut and format a cell in the html table

In [15]:
def get_value(c, n, v) :
    return lookup[c].loc[n][v]

def format_cell (df,c,add_id=False,add_plus=False,add_minus=False) :

    divs = pd.DataFrame(index=df.index)
    
    pattern = '<div class="circle" style="background-color:%s"></div>'
    divs['circle'] = [pattern % get_value(c,x,'color_hex_triplet') for x in df[c]]
    
    pattern = '<div class="celltext">%s</div>'
    divs['name'] = [pattern % x for x in df[c]]
   
    divs['id'] = ''
    if add_id :
        pattern = '<div id="%s"></div>'
        divs['id'] = [pattern % get_value(c,x,'label') for x in df[c]]
        
    divs['plus'] = ''
    if add_plus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[+]</a></div>'
        divs['plus'] = [pattern % (get_value(c,x,'first_child_term_set_name'),
                                   get_value(c,x,'first_child_label')) for x in df[c]]
        
    divs['minus'] = ''
    if add_minus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[-]</a></div>'
        divs['minus'] = [pattern % (get_value(c,x,'cluster_annotation_term_set_name'),
                                    get_value(c,x,'label')) for x in df[c]]
    
    cols = ['id','circle','name','plus','minus']
    output = divs[cols].apply(lambda row: ''.join(row.values.astype(str)), axis=1)
    return output


Helper function to create html document

In [16]:
def create_html(df, ts, file, title):
    
    # apply formatter to each term set
    df_formatted = df.copy()
    
    for tag in term_sets['name'] :
        if tag in df_formatted.columns :

            add_id = False
            if tag == ts :
                add_id = True
                
            add_plus = False
            if tag == ts and tag not in ['Supertype']:
                add_plus = True
                
            add_minus = False
            if tag != ts and tag not in ['']:
                add_minus = True
                
            df_formatted[tag] = format_cell(df,tag,add_id,add_plus,add_minus)
            
            
    output = df_formatted.to_html(index=False, na_rep='',
                        render_links=True,escape=False,
                        classes="mystyle")

    html_string = '''
    <html>
    <head><title>%s</title></head>
    <link rel="stylesheet" type="text/css" href="../../simple_style.css"/>
    <body>
    {table}
    </body>
    </html>.
    ''' % title

    # OUTPUT AN HTML FILE
    with open(file, 'w') as f:
        f.write(html_string.format(table=output))

In [17]:
# Write the data to the _static directory of the abc_atlas_access so that links work properly in the jupyter-book/sphinx page.
output_directory = os.path.join('../../_static', taxonomy_dir, version)
os.makedirs(output_directory, exist_ok=True)

In [18]:
df_supertype = pivot[['Class']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'class.html')
title = 'SEA-AD Multiregion taxonomy: Class'
create_html(df_supertype, 'Class', file, title)
print(len(df_supertype))

3


In [19]:
df_supertype = pivot[['Class', 'Subclass']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'subclass.html')
title = 'SEA-AD Multiregion taxonomy: Subclass'
create_html(df_supertype, 'Subclass', file, title)
print(len(df_supertype))

29


In [20]:
df_supertype = pivot[['Class', 'Subclass', 'Supertype']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'supertype.html')
title = 'SEA-AD Multiregion taxonomy: Supertype'
create_html(df_supertype, 'Supertype', file, title)
print(len(df_supertype))

207
